# Bankruptcy Prediction Pipeline
**Dataset:** Polish Companies Bankruptcy (UCI)
**Model:** XGBoost with Optuna hyperparameter tuning
**Goal:** Predict bankruptcy 2 years ahead (3year dataset)

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import os
import json
import warnings
warnings.filterwarnings('ignore')

from scipy.io import arff
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, fbeta_score,
    average_precision_score, matthews_corrcoef,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Set working directory
os.chdir(r'C:\Users\izanama\Documents\GitHub\bancrucy')
print('Setup complete!')

## 2. Data Loading & Missing Value Imputation (MICE + Random Forest)

In [ ]:
from sklearn.exceptions import ConvergenceWarning

os.makedirs('imputed_data', exist_ok=True)

for year in ['1year', '2year', '3year', '4year', '5year']:
    data, meta = arff.loadarff(f'{year}.arff')
    df_year = pd.DataFrame(data)
    X_year = df_year.drop(columns=['class'])
    y_year = df_year['class']

    # Winsorization (clip outliers at 1%-99%)
    for col in X_year.columns:
        low  = X_year[col].quantile(0.01)
        high = X_year[col].quantile(0.99)
        X_year[col] = X_year[col].clip(lower=low, upper=high)

    # MICE imputation with Random Forest
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        imputer = IterativeImputer(
            estimator=RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1),
            max_iter=5, random_state=42, verbose=0
        )
        X_imputed = pd.DataFrame(imputer.fit_transform(X_year), columns=X_year.columns)

    X_imputed['class'] = y_year.values
    X_imputed.to_csv(f'{year}_imputed.csv', index=False)
    print(f'{year} done, missing values left: {X_imputed.isna().sum().sum()}')

print('\nAll datasets imputed and saved!')

## 3. Hyperparameter Tuning with Optuna (F2 Score)

In [ ]:
from sklearn.metrics import fbeta_score, make_scorer

f2_scorer = make_scorer(fbeta_score, beta=2)
best_params_per_year = {}

for year in ['1year', '2year', '3year', '4year', '5year']:
    print(f'Optimizing {year}...')

    df = pd.read_csv(f'{year}_imputed.csv')
    df['class'] = df['class'].str.replace("b'", '').str.replace("'", '').astype(int)
    X = df.drop(columns=['class'])
    y = df['class']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 100, 1000),
            'max_depth':         trial.suggest_int('max_depth', 3, 10),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
            'gamma':             trial.suggest_float('gamma', 0, 5),
            'scale_pos_weight':  trial.suggest_float('scale_pos_weight', scale_pos_weight, scale_pos_weight * 3),
            'device':            'cuda',
            'random_state':      42,
        }
        model = XGBClassifier(**params)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=f2_scorer)
        return scores.mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    best_params_per_year[year] = study.best_params
    print(f'{year} best F2: {study.best_value:.4f}\n')

# Save params to file
with open('best_params.json', 'w') as f:
    json.dump(best_params_per_year, f)
print('Best params saved to best_params.json!')

## 4. Load Saved Params (skip Optuna if already run)

In [ ]:
with open('best_params.json', 'r') as f:
    best_params_per_year = json.load(f)
print('Params loaded!')
print(best_params_per_year['3year'])

## 5. Model Performance Across All Years

In [ ]:
print(f"{'Year':<8} {'ROC-AUC':<10} {'Precision':<12} {'Recall':<10} {'F1':<8} {'F2':<8}")
print('=' * 58)

for year in ['1year', '2year', '3year', '4year', '5year']:
    df = pd.read_csv(f'{year}_imputed.csv')
    df['class'] = df['class'].str.replace("b'", '').str.replace("'", '').astype(int)
    X = df.drop(columns=['class'])
    y = df['class']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    model = XGBClassifier(**best_params_per_year[year], device='cuda', random_state=42)
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= 0.1).astype(int)

    auc = roc_auc_score(y_test, y_proba)
    p   = precision_score(y_test, y_pred)
    r   = recall_score(y_test, y_pred)
    f1  = fbeta_score(y_test, y_pred, beta=1)
    f2  = fbeta_score(y_test, y_pred, beta=2)

    print(f'{year:<8} {auc:<10.4f} {p:<12.4f} {r:<10.4f} {f1:<8.4f} {f2:<8.4f}')

## 6. Final Model — 3year Dataset (2-year Prediction Horizon)

In [ ]:
df = pd.read_csv('3year_imputed.csv')
df['class'] = df['class'].str.replace("b'", '').str.replace("'", '').astype(int)
X = df.drop(columns=['class'])
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost
model_xgb = XGBClassifier(**best_params_per_year['3year'], device='cuda', random_state=42)
model_xgb.fit(X_train, y_train)
y_proba     = model_xgb.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba >= 0.1).astype(int)

# Random Forest (baseline)
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight={0: 1, 1: scale_pos_weight},
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf  = (y_proba_rf >= 0.1).astype(int)

print('=== Final Model Summary (threshold = 0.1) ===')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_final):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_final):.4f}')
print(f'F2:        {fbeta_score(y_test, y_pred_final, beta=2):.4f}')
print(f'MCC:       {matthews_corrcoef(y_test, y_pred_final):.4f}')
print(f'\nOut of {len(y_test)} companies:')
print(f'  Flagged for review: {y_pred_final.sum()}')
print(f'  Cleared:            {(y_pred_final==0).sum()}')
print(f'  Bankrupts found:    {((y_pred_final==1) & (y_test==1)).sum()} out of {y_test.sum()}')

## 7. Threshold Analysis

In [ ]:
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<10} {'F2':<8}")
print('=' * 44)
for threshold in [i/100 for i in range(5, 95, 5)]:
    y_pred_t = (y_proba >= threshold).astype(int)
    if y_pred_t.sum() == 0:
        break
    p  = precision_score(y_test, y_pred_t)
    r  = recall_score(y_test, y_pred_t)
    f2 = fbeta_score(y_test, y_pred_t, beta=2)
    print(f'{threshold:<12} {p:<12.4f} {r:<10.4f} {f2:<8.4f}')

## 8. Visualizations

In [ ]:
def compute_lift(y_true, y_proba):
    df_lift = pd.DataFrame({'true': y_true.values, 'proba': y_proba})
    df_lift = df_lift.sort_values('proba', ascending=False).reset_index(drop=True)
    df_lift['cumulative_pos'] = df_lift['true'].cumsum()
    df_lift['cumulative_pct'] = (df_lift.index + 1) / len(df_lift)
    df_lift['lift'] = (df_lift['cumulative_pos'] / (df_lift.index + 1)) / df_lift['true'].mean()
    return df_lift['cumulative_pct'], df_lift['lift']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. ROC Curve
RocCurveDisplay.from_predictions(y_test, y_proba,    ax=axes[0,0], name='XGBoost')
RocCurveDisplay.from_predictions(y_test, y_proba_rf, ax=axes[0,0], name='Random Forest')
axes[0,0].set_title('ROC Curve')

# 2. Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(y_test, y_proba,    ax=axes[0,1], name='XGBoost')
PrecisionRecallDisplay.from_predictions(y_test, y_proba_rf, ax=axes[0,1], name='Random Forest')
axes[0,1].set_title('Precision-Recall Curve')

# 3. Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final,
    display_labels=['Not Bankrupt', 'Bankrupt'],
    ax=axes[0,2]
)
axes[0,2].set_title('Confusion Matrix XGBoost (threshold 0.1)')

# 4. Probability Distribution
axes[1,0].hist(y_proba[y_test==0], bins=50, alpha=0.5, label='Not Bankrupt', color='green')
axes[1,0].hist(y_proba[y_test==1], bins=50, alpha=0.5, label='Bankrupt',     color='red')
axes[1,0].axvline(x=0.1, color='black', linestyle='--', label='Threshold 0.1')
axes[1,0].legend()
axes[1,0].set_title('Predicted Probability Distribution')
axes[1,0].set_xlabel('Bankruptcy Probability')
axes[1,0].set_ylabel('Count')

# 5. Lift Curve
pct_xgb, lift_xgb = compute_lift(y_test, y_proba)
pct_rf,  lift_rf  = compute_lift(y_test, y_proba_rf)
axes[1,1].plot(pct_xgb, lift_xgb, label='XGBoost',      color='blue')
axes[1,1].plot(pct_rf,  lift_rf,  label='Random Forest', color='orange')
axes[1,1].axhline(y=1, color='black', linestyle='--', label='Baseline')
axes[1,1].set_title('Lift Curve')
axes[1,1].set_xlabel('% of Companies (sorted by probability)')
axes[1,1].set_ylabel('Lift')
axes[1,1].legend()

# 6. Metrics Comparison
metrics = {
    'ROC-AUC':   [roc_auc_score(y_test, y_proba),        roc_auc_score(y_test, y_proba_rf)],
    'Recall':    [recall_score(y_test, y_pred_final),     recall_score(y_test, y_pred_rf)],
    'Precision': [precision_score(y_test, y_pred_final),  precision_score(y_test, y_pred_rf)],
    'F2':        [fbeta_score(y_test, y_pred_final, beta=2), fbeta_score(y_test, y_pred_rf, beta=2)],
    'MCC':       [matthews_corrcoef(y_test, y_pred_final), matthews_corrcoef(y_test, y_pred_rf)],
}
x = np.arange(len(metrics))
axes[1,2].bar(x - 0.2, [v[0] for v in metrics.values()], 0.4, label='XGBoost',      color='blue')
axes[1,2].bar(x + 0.2, [v[1] for v in metrics.values()], 0.4, label='Random Forest', color='orange')
axes[1,2].set_xticks(x)
axes[1,2].set_xticklabels(metrics.keys())
axes[1,2].set_title('XGBoost vs Random Forest')
axes[1,2].legend()

plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. SHAP Feature Importance

In [ ]:
feature_names = {
    'Attr1': 'Net profit / Total assets',
    'Attr2': 'Total liabilities / Total assets',
    'Attr3': 'Working capital / Total assets',
    'Attr4': 'Current assets / ST liabilities',
    'Attr5': 'Quick liquidity ratio * 365',
    'Attr6': 'Retained earnings / Total assets',
    'Attr7': 'EBIT / Total assets',
    'Attr8': 'Book value of equity / Total liabilities',
    'Attr9': 'Sales / Total assets',
    'Attr10': 'Equity / Total assets',
    'Attr11': '(GP + Extraordinary + Fin exp) / Total assets',
    'Attr12': 'Gross profit / ST liabilities',
    'Attr13': '(Gross profit + Depreciation) / Sales',
    'Attr14': '(Gross profit + Interest) / Total assets',
    'Attr15': '(Total liabilities * 365) / (GP + Dep)',
    'Attr16': '(Gross profit + Depreciation) / Total liabilities',
    'Attr17': 'Total assets / Total liabilities',
    'Attr18': 'Gross profit / Total assets',
    'Attr19': 'Gross profit / Sales',
    'Attr20': '(Inventory * 365) / Sales',
    'Attr21': 'Sales growth (n/n-1)',
    'Attr22': 'Operating profit / Total assets',
    'Attr23': 'Net profit / Sales',
    'Attr24': '3-year Gross profit / Total assets',
    'Attr25': '(Equity - Share capital) / Total assets',
    'Attr26': '(Net profit + Depreciation) / Total liabilities',
    'Attr27': 'Operating profit / Financial expenses',
    'Attr28': 'Working capital / Fixed assets',
    'Attr29': 'Log of Total assets',
    'Attr30': '(Total liabilities - Cash) / Sales',
    'Attr31': '(Gross profit + Interest) / Sales',
    'Attr32': '(Current liabilities * 365) / COGS',
    'Attr33': 'Operating expenses / ST liabilities',
    'Attr34': 'Operating expenses / Total liabilities',
    'Attr35': 'Profit on sales / Total assets',
    'Attr36': 'Total sales / Total assets',
    'Attr37': '(Current assets - Inventories) / LT liabilities',
    'Attr38': 'Constant capital / Total assets',
    'Attr39': 'Profit on sales / Sales',
    'Attr40': '(Current assets - Inv - Rec) / ST liabilities',
    'Attr41': 'Total liabilities / (Op profit + Dep) * 365/12',
    'Attr42': 'Operating profit / Sales',
    'Attr43': 'Receivables + Inventory turnover days',
    'Attr44': '(Receivables * 365) / Sales',
    'Attr45': 'Net profit / Inventory',
    'Attr46': '(Current assets - Inventory) / ST liabilities',
    'Attr47': '(Inventory * 365) / COGS',
    'Attr48': 'EBITDA / Total assets',
    'Attr49': 'EBITDA / Sales',
    'Attr50': 'Current assets / Total liabilities',
    'Attr51': 'ST liabilities / Total assets',
    'Attr52': '(ST liabilities * 365) / COGS',
    'Attr53': 'Equity / Fixed assets',
    'Attr54': 'Constant capital / Fixed assets',
    'Attr55': 'Working capital',
    'Attr56': '(Sales - COGS) / Sales',
    'Attr57': '(CA - Inv - ST liab) / (Sales - GP - Dep)',
    'Attr58': 'Total costs / Total sales',
    'Attr59': 'LT liabilities / Equity',
    'Attr60': 'Sales / Inventory',
    'Attr61': 'Sales / Receivables',
    'Attr62': '(ST liabilities * 365) / Sales',
    'Attr63': 'Sales / ST liabilities',
    'Attr64': 'Sales / Fixed assets'
}

X_test_named = X_test.rename(columns=feature_names)
explainer    = shap.TreeExplainer(model_xgb)
shap_values  = explainer.shap_values(X_test_named)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_named, plot_type='bar', max_display=15, show=False)
plt.title('Feature Importance (SHAP) — Top 15')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_named, max_display=15, show=False)
plt.title('SHAP Feature Impact — Top 15')
plt.tight_layout()
plt.savefig('shap_impact.png', dpi=150, bbox_inches='tight')
plt.show()